# Sesión 1. De los datos del Combine a una base de análisis

**¿Qué características de un jugador permiten pronosticar su éxito en la NFL?**

*Programar con criterio en la era de la IA · Proyecto: pronosticar el éxito en la NFL · Versión para Google Colab*

> **Esta es la versión para Google Colab**, para quien no pueda instalar R en su computadora. Funciona en el navegador, sin instalar nada. El contenido es el mismo que el del archivo `.Rmd`.
>
> 1. **Comprueba que el cuaderno usa R:** menú *Entorno de ejecución > Cambiar tipo de entorno de ejecución*, y en *Tipo de entorno de ejecución* elige **R**. Normalmente ya viene así.
> 2. **Guarda tu copia:** *Archivo > Guardar una copia en Drive*. Trabaja siempre en tu copia; si no, tus cambios se pierden al cerrar la pestaña.
> 3. **Ejecuta la celda de preparación** (la siguiente) antes que cualquier otra. Si tiene que instalar paquetes, tarda uno o dos minutos.
> 4. Colab se reinicia si lo dejas un rato sin usar: si una celda dice que no encuentra un objeto, usa *Entorno de ejecución > Ejecutar anteriores*.

In [ ]:
#@title Preparación: ejecuta esta celda antes que las demás
# Comprueba que el cuaderno usa R e instala los paquetes del curso que falten.
paquetes <- c("dplyr", "ggplot2")
faltan <- paquetes[!sapply(paquetes, requireNamespace, quietly = TRUE)]
if (length(faltan) > 0) install.packages(faltan)
options(width = 100, repr.plot.res = 110, crayon.enabled = FALSE, cli.num_colors = 1)
cat(R.version.string, "· paquetes listos:", paste(paquetes, collapse = ", "), "\n")

# Para empezar: de qué trata este curso

En este curso vas a aprender a programar en R trabajando en un solo proyecto, de principio a fin. El tema es el fútbol americano profesional de Estados Unidos, la **NFL**, pero no necesitas saber nada de fútbol americano: todo lo que hace falta está en esta sección. Lo que sí vas a reconocer es el problema de fondo, porque es un problema de economía.

## Un problema de economía con casco

Cada primavera, los 32 equipos de la NFL contratan a unos 250 jugadores que acaban de terminar su etapa universitaria. Es una decisión cara y con información incompleta. El equipo ve **señales**: cuánto mide el jugador, qué tan rápido corre, qué hizo en la universidad, cómo le fue en una entrevista. Pero no ve lo que de verdad le importa: **qué tan buen profesional será**. Es el mismo problema de una empresa que contrata a recién egresados mirando sus calificaciones, o de un banco que decide un crédito mirando el historial de quien lo pide.

La NFL tiene algo que casi ningún otro mercado de trabajo tiene: **datos públicos de todo el proceso**. Sabemos cómo le fue a cada candidato en las pruebas, quién lo contrató, en qué orden y cómo fue su carrera en los diez o quince años siguientes. Eso permite hacer lo que haremos en el curso: construir un pronóstico y **comprobar** si acierta.

## El fútbol americano en dos minutos

Se enfrentan dos equipos de 11 jugadores. El equipo que tiene el balón, el **ataque**, intenta avanzar hasta la zona de anotación del rival corriendo con el balón o lanzando pases. El otro, la **defensa**, intenta impedirlo. Cada equipo tiene jugadores distintos para atacar y para defender, y cada jugador ocupa una **posición** con una tarea muy concreta. En los datos, cada jugador tiene un código de posición, `simple_pos`:

| Código | Posición | Qué hace |
|--------|----------|----------|
| `QB` | Mariscal de campo (*quarterback*) | Dirige el ataque y lanza los pases |
| `RB` | Corredor (*running back*) | Corre con el balón; rápido y difícil de derribar |
| `WR` | Receptor (*wide receiver*) | Atrapa los pases; suelen ser los más rápidos |
| `TE` | Ala cerrada (*tight end*) | Atrapa pases y también bloquea; grande y rápido a la vez |
| `OL` | Línea ofensiva | Bloquea para proteger al mariscal y abrirle paso al corredor; los más pesados del campo |
| `DL` | Línea defensiva | Empuja contra la línea ofensiva para llegar al mariscal o al corredor |
| `LB` | Apoyador (*linebacker*) | Detiene al corredor y cubre pases cortos |
| `DB` | Defensivo profundo (esquineros y profundos) | Cubre a los receptores e intercepta pases |
| `ST` | Equipos especiales | Patean el balón; los dejaremos fuera del análisis |

No hace falta memorizarla: vuelve aquí cada vez que aparezca un código.

## Cómo llega un jugador a la NFL

| Cuándo | Etapa | Qué pasa |
|--------|-------|----------|
| Tres a cinco años | **1. La universidad** | Los jugadores se forman en el fútbol universitario, la **NCAA**, que en Estados Unidos es enorme. Ahí acumulan estadísticas: yardas, pases completos, tacleadas. |
| Febrero | **2. El Combine** | La NFL invita a unos 300 prospectos a una semana de pruebas: los miden, los pesan, los cronometran en 40 yardas y los hacen saltar, levantar pesas y cambiar de dirección. |
| Abril | **3. El draft** | Los 32 equipos eligen jugadores por turnos durante 7 rondas; en general, el que tuvo la peor temporada elige primero. El orden en que alguien es elegido es su **número de selección**: 1 es el primero; alrededor de 250, el último. |
| Los años siguientes | **4. La carrera** | Algunos juegan quince temporadas y van varias veces al partido de las estrellas; otros nunca juegan un partido oficial. |

Un jugador que no es elegido en el draft puede firmar después con algún equipo, pero en nuestros datos no hay información de su carrera.

## ¿Cómo se mide el éxito de una carrera?

No es obvio: un buen mariscal acumula miles de yardas, mientras que un buen liniero ofensivo casi no aparece en las estadísticas. Usaremos dos medidas:

- **Valor de carrera** (`career_av`). El sitio Pro Football Reference le asigna a cada jugador, en cada temporada, un número llamado *Approximate Value* (AV) que resume lo que aportó: si fue titular, sus estadísticas y el rendimiento de su unidad. La suma de todas sus temporadas es su valor de carrera. Para darte una idea, entre los elegidos de 2000 a 2014 la mitad acumuló 8 puntos o menos, uno de cada cinco se quedó en 0, uno de cada diez pasó de 50 y muy pocos superaron los 100. Drew Brees tiene 167; Tom Brady, 184, el máximo del archivo.
- **Pro Bowl**, el partido de las estrellas de la NFL. Ir al menos una vez es una distinción que alcanzó uno de cada ocho elegidos.

## La pregunta del curso

En el curso te pondrás en el lugar del departamento de análisis de un equipo y responderás una pregunta concreta:

> ¿Qué características de un jugador, conocidas antes o en el momento del draft, permiten pronosticar su éxito futuro en la NFL?

## La historia del curso en tres gráficas

Antes de escribir una línea de código, mira adónde vamos a llegar. Estas tres gráficas resumen lo que encontrarás en el curso. Todavía no necesitas entender cómo se hicieron.

In [ ]:
#@title Preparar las gráficas de la portada (no hace falta entender este código)
# En las sesiones 3, 4 y 5 escribirás código parecido a este.
hay_graficas <- requireNamespace("ggplot2", quietly = TRUE)
url_portada <- paste0("https://raw.githubusercontent.com/phcs971/nfl-draft-dataset/",
                      "54730eaae524dbff0647e966cd12c739724f807f/nfl_data.csv")
portada <- tryCatch(read.csv(url_portada, sep = ";", na.strings = "", encoding = "UTF-8"),
                    error = function(e) NULL)
if (is.null(portada) && file.exists("datos/nfl_data.csv")) {
  portada <- read.csv("datos/nfl_data.csv", sep = ";", na.strings = "", encoding = "UTF-8")
}
hay_graficas <- hay_graficas && !is.null(portada)

if (hay_graficas) {
  suppressPackageStartupMessages({library(dplyr); library(ggplot2)})
  azul <- "#2a78d6"; naranja <- "#eb6834"; gris <- "#52514e"; tenue <- "#898781"
  fuente <- "Datos: phcs971/nfl-draft-dataset (Combine, draft, Pro Football Reference y Sports Reference). Sin pateadores ni despejadores."
  mil <- function(x) format(x, big.mark = ",")
  estilo_portada <- theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 16), plot.title.position = "plot",
          plot.subtitle = element_text(color = gris, size = 12.5, margin = margin(b = 10)),
          plot.caption = element_text(color = tenue, size = 9.5, hjust = 0, lineheight = 1.2,
                                      margin = margin(t = 12)),
          plot.caption.position = "plot",
          panel.grid.minor = element_blank(), panel.grid.major = element_line(color = "#e8e7e1"),
          strip.text = element_text(face = "bold", hjust = 0, size = 12.5), strip.clip = "off",
          axis.title = element_text(color = gris), axis.text = element_text(size = 11, color = gris),
          legend.position = "top", legend.justification = "left", legend.text = element_text(color = gris),
          plot.margin = margin(12, 18, 10, 18))

  repetidos <- portada$tag[duplicated(portada$tag)]
  todos <- portada |>
    rename(forty = X40_yard) |>
    mutate(drafted = as.logical(drafted)) |>
    filter(year >= 1987, year <= 2014, simple_pos != "ST", !(tag %in% repetidos))

  # 1. Puntaje del Combine y ronda en que se elige -----------------------------
  pruebas_signo <- c(forty = -1, bench_press = 1, vert_leap = 1, broad_jump = 1, shuttle = -1)
  z <- function(x) (x - mean(x, na.rm = TRUE)) / sd(x, na.rm = TRUE)
  d1 <- todos |>
    group_by(simple_pos, year) |>
    mutate(across(all_of(names(pruebas_signo)), z, .names = "z_{.col}")) |>
    ungroup()
  zs <- sweep(as.matrix(d1[paste0("z_", names(pruebas_signo))]), 2, pruebas_signo, "*")
  d1$puntaje <- rowMeans(zs, na.rm = TRUE)
  sin_pruebas <- sum(is.nan(d1$puntaje))
  paneles1 <- c(paste("Elegido en la ronda", 1:6), "Elegido en la ronda 7 o más", "No elegido")
  d1 <- d1 |>
    filter(!is.nan(puntaje)) |>
    mutate(grupo = 11 - ntile(puntaje, 10),   # 1 = el 10 % con mejor resultado
           ronda = case_when(!drafted ~ "No elegido",
                             draft_round >= 7 ~ "Elegido en la ronda 7 o más",
                             TRUE ~ paste("Elegido en la ronda", draft_round)))
  p1 <- d1 |>
    count(grupo, ronda) |>
    group_by(grupo) |>
    mutate(prob = n / sum(n)) |>
    ungroup() |>
    mutate(ronda = factor(ronda, levels = paneles1), tipo = ifelse(ronda == "No elegido", "no", "si"))
  extremos <- p1 |>
    group_by(ronda) |>
    mutate(y_et = sapply(grupo, function(g) max(prob[grupo %in% c(g, g + ifelse(g == 1, 1, -1))]))) |>
    ungroup() |>
    filter(grupo %in% c(1, 10))
  topes <- data.frame(ronda = factor(paneles1, levels = paneles1), grupo = 1, prob = c(rep(0.30, 7), 0.60))

  grafica_1 <- ggplot(p1, aes(grupo, prob)) +
    geom_blank(data = topes) +
    geom_col(aes(fill = tipo), width = 0.78) +
    geom_text(data = extremos, aes(y = y_et, label = paste0(round(100 * prob), "%")),
              vjust = -0.45, size = 3.6, color = "#0b0b0b", fontface = "bold") +
    facet_wrap(~ ronda, nrow = 2, scales = "free_y",
               labeller = as_labeller(setNames(c(paneles1[1:7], "No elegido (eje hasta 60 %)"), paneles1))) +
    scale_fill_manual(values = c(si = azul, no = tenue), guide = "none") +
    scale_x_continuous(breaks = c(1, 10), labels = c("Mejor\n10 %", "Peor\n10 %"), expand = expansion(add = 0.6)) +
    scale_y_continuous(labels = function(x) paste0(round(100 * x), "%"),
                       breaks = function(l) if (l[2] > 0.4) seq(0, 0.6, 0.2) else seq(0, 0.3, 0.1),
                       expand = expansion(mult = c(0, 0.08))) +
    labs(title = "A mejor resultado en el Combine, más probable ser elegido en las primeras rondas",
         subtitle = paste0("Porcentaje de jugadores elegidos en cada ronda según su resultado en el Combine · ",
                           mil(nrow(d1)), " participantes, 1987–2014"),
         x = "Resultado en el Combine: jugadores ordenados del mejor al peor, en 10 grupos del mismo tamaño",
         y = "Porcentaje de jugadores del grupo",
         caption = paste0(
           "Cada barra es un grupo de unos ", round(nrow(d1) / 10, -1), " jugadores; su altura es el porcentaje de ese grupo que fue elegido en esa ronda. Las ocho barras de un mismo grupo suman 100 %.\n",
           "Resultado en el Combine: promedio de las pruebas que hizo cada jugador (40 yardas, press de banca, salto vertical, salto largo y shuttle), cada una normalizada frente\n",
           "a los de su misma posición y año, y con el signo cambiado en las pruebas de tiempo para que más sea siempre mejor. Las medidas corporales no entran porque más no\n",
           "siempre es mejor. Hasta 1993 el draft tuvo más de 7 rondas; esas rondas se suman a la 7. No aparecen ", mil(sin_pruebas), " participantes que no hicieron ninguna prueba.\n",
           fuente)) +
    estilo_portada +
    theme(panel.grid.major.x = element_blank(), panel.spacing.x = unit(1.6, "lines"),
          panel.spacing.y = unit(1.4, "lines"), axis.title.x = element_text(margin = margin(t = 8)))

  # 2. Número de selección y valor de la carrera -------------------------------
  e <- todos |> filter(drafted) |> mutate(av = ifelse(is.na(career_av), 0, career_av))
  prom <- e |>
    mutate(grupo = floor((draft_pick - 1) / 8)) |>
    group_by(grupo) |>
    summarise(pick = mean(draft_pick), av = mean(av), n = n(), .groups = "drop") |>
    filter(n >= 10)
  destacados <- e |> filter(name %in% c("Tom Brady", "Peyton Manning"))
  grafica_2 <- ggplot(e, aes(draft_pick, av)) +
    geom_point(aes(color = "Cada jugador elegido"), position = position_jitter(width = 0.4, height = 0.3, seed = 2),
               size = 0.9, alpha = 0.28, shape = 16) +
    geom_line(data = prom, aes(pick, av, color = "Promedio cada 8 selecciones"), linewidth = 1.1) +
    geom_point(data = prom, aes(pick, av, color = "Promedio cada 8 selecciones"), size = 1.8) +
    geom_point(data = destacados, color = "#0b0b0b", size = 3) +
    geom_text(data = destacados, aes(label = name), hjust = -0.12, vjust = -0.6, size = 3.8,
              color = "#0b0b0b", fontface = "bold") +
    scale_color_manual(values = c("Cada jugador elegido" = azul, "Promedio cada 8 selecciones" = naranja), name = NULL) +
    guides(color = guide_legend(override.aes = list(size = 3, alpha = 1, linetype = c(0, 1)))) +
    scale_x_continuous(breaks = c(1, 32, 64, 100, 150, 200, 250, 300)) +
    scale_y_continuous(breaks = seq(0, 175, 25)) +
    labs(title = "Las primeras selecciones tienen mejores carreras en promedio, pero hay enormes excepciones",
         subtitle = paste0("Valor de carrera según el número de selección · ", mil(nrow(e)), " jugadores elegidos, 1987–2014"),
         x = "Número de selección en el draft", y = "Valor de carrera (career_av)",
         caption = paste0("Los jugadores elegidos que nunca jugaron cuentan con valor 0. Hasta 1993 el draft tuvo más de 7 rondas: por eso hay selecciones después de la 262.\n", fuente)) +
    estilo_portada

  # 3. Pronóstico antes y después del draft --------------------------------------
  defensa <- c("DL", "LB", "DB")
  elegidos <- todos |>
    filter(drafted, year >= 2000) |>
    mutate(career_av = ifelse(is.na(career_av), 0, career_av),
           ncaa_yardas_pase      = ifelse(simple_pos == "QB", log1p(pmax(college_pass_yds, 0)), 0),
           ncaa_td_pase          = ifelse(simple_pos == "QB", log1p(college_pass_td), 0),
           ncaa_yardas_tierra    = ifelse(simple_pos == "RB", log1p(pmax(college_rush_yds, 0)), 0),
           ncaa_yardas_recepcion = ifelse(simple_pos %in% c("WR", "TE"), log1p(pmax(college_rec_yds, 0)), 0),
           ncaa_tacleadas        = ifelse(simple_pos %in% defensa, log1p(college_tackles), 0),
           ncaa_capturas         = ifelse(simple_pos %in% defensa, log1p(college_sacks), 0),
           ncaa_intercepciones   = ifelse(simple_pos %in% defensa, log1p(college_ints), 0),
           ncaa_partidos         = college_games)
  pruebas <- c("forty", "vert_leap", "broad_jump", "bench_press", "shuttle")
  estadisticas_ncaa <- c("ncaa_yardas_pase", "ncaa_td_pase", "ncaa_yardas_tierra", "ncaa_yardas_recepcion",
                         "ncaa_tacleadas", "ncaa_capturas", "ncaa_intercepciones")
  marcar_y_rellenar <- function(datos, aprende) {
    datos <- datos |>
      mutate(across(all_of(pruebas), is.na, .names = "sin_{.col}"),
             sin_ncaa     = if_any(all_of(estadisticas_ncaa), is.na),
             sin_partidos = is.na(ncaa_partidos))
    for (v in c(pruebas, estadisticas_ncaa, "ncaa_partidos")) {
      medianas <- tapply(aprende[[v]], aprende$simple_pos, median, na.rm = TRUE)
      datos[[v]] <- ifelse(is.na(datos[[v]]), medianas[datos$simple_pos], datos[[v]])
    }
    datos
  }
  ent <- elegidos |> filter(year <= 2010)
  pru <- elegidos |> filter(year >= 2011) |> marcar_y_rellenar(aprende = ent)
  ent <- ent |> marcar_y_rellenar(aprende = ent)
  ent$raiz_av <- sqrt(pmax(ent$career_av, 0))
  predictores_pre <- c("simple_pos", "height", "weight", pruebas, paste0("sin_", pruebas),
                       estadisticas_ncaa, "ncaa_partidos", "sin_ncaa", "sin_partidos")
  modelos <- list("Antes del draft"   = reformulate(predictores_pre, response = "raiz_av"),
                  "Después del draft" = reformulate(c(predictores_pre, "log(draft_pick)"), response = "raiz_av"))
  p3 <- bind_rows(lapply(names(modelos), function(m) {
    ajuste <- lm(modelos[[m]], data = ent)
    data.frame(momento = m, pronostico = pmax(predict(ajuste, pru), 0)^2, real = pmax(pru$career_av, 0),
               error = abs(pru$career_av - pmax(predict(ajuste, pru), 0)^2))
  })) |>
    group_by(momento) |>
    mutate(panel = paste0(momento, " · error medio: ", format(round(mean(error), 1), nsmall = 1), " puntos")) |>
    ungroup()
  decimos <- p3 |>
    group_by(panel) |>
    mutate(decimo = ntile(pronostico, 10)) |>
    group_by(panel, decimo) |>
    summarise(pronostico = mean(pronostico), promedio = mean(real), mediana = median(real), .groups = "drop")
  marcas <- bind_rows(decimos |> transmute(panel, pronostico, valor = promedio, tipo = "Promedio real de cada décimo"),
                      decimos |> transmute(panel, pronostico, valor = mediana, tipo = "Mediana real de cada décimo"))
  cortes <- c(0, 5, 25, 50, 100, 150)
  grafica_3 <- ggplot(p3, aes(pronostico, real)) +
    geom_abline(aes(slope = 1, intercept = 0, linetype = "Pronóstico perfecto"), color = gris) +
    geom_point(color = azul, alpha = 0.25, size = 1.3, shape = 16) +
    geom_point(data = marcas, aes(pronostico, valor, shape = tipo), color = naranja, fill = "white",
               size = 3.2, stroke = 1.3) +
    facet_wrap(~ panel) +
    scale_shape_manual(values = c("Promedio real de cada décimo" = 16, "Mediana real de cada décimo" = 21),
                       breaks = c("Promedio real de cada décimo", "Mediana real de cada décimo"), name = NULL) +
    scale_linetype_manual(values = c("Pronóstico perfecto" = "dashed"), name = NULL) +
    scale_x_sqrt(breaks = cortes, limits = c(0, 190)) +
    scale_y_sqrt(breaks = cortes, limits = c(0, 190)) +
    coord_equal() +
    labs(title = "Más información mejora el pronóstico, pero el ruido sigue siendo enorme",
         subtitle = paste0("Valor de carrera pronosticado y real · ", mil(nrow(pru)), " jugadores elegidos en 2011–2014"),
         x = "Valor de carrera pronosticado", y = "Valor de carrera real",
         caption = paste0(
           "Regresión lineal de la raíz cuadrada del valor de carrera, estimada con los elegidos de 2000–2010 y evaluada en los de 2011–2014, que el modelo no vio.\n",
           "Antes del draft: posición, medidas y pruebas del Combine y estadísticas universitarias (NCAA). Después del draft: lo mismo más el número de selección.\n",
           "Los datos faltantes se rellenan con la mediana de la posición, con un indicador de dato faltante. Error medio: promedio de la diferencia absoluta entre\n",
           "lo pronosticado y lo real. Naranja: jugadores ordenados por su pronóstico en 10 grupos del mismo tamaño. Ejes en escala de raíz cuadrada.\n", fuente)) +
    estilo_portada +
    theme(panel.spacing.x = unit(2, "lines"))
}

sin_grafica <- function() {
  plot.new()
  text(0.5, 0.5, "Sin conexión: la gráfica aparecerá cuando R pueda leer los datos (sección 2)\no cuando esté instalado el paquete ggplot2 (sección «Antes de empezar»).")
}

**Primero, el Combine.** ¿A quienes mejor salen en las pruebas les va mejor en el draft? Ordenamos a los participantes del Combine de mejor a peor resultado y los repartimos en diez grupos del mismo tamaño. Cada panel muestra qué porcentaje de cada grupo fue elegido en una ronda; el último, qué porcentaje no fue elegido.

In [ ]:
#@title Gráfica 1
options(repr.plot.width = 12, repr.plot.height = 8)
if (hay_graficas) print(grafica_1) else sin_grafica()

Entre el 10 % con mejor Combine, uno de cada cinco fue elegido en la primera ronda; entre el 10 % con peor resultado, dos de cada cien. Con los no elegidos pasa al revés. **El Combine ayuda a pronosticar quién será elegido y cuándo**, pero no lo decide: incluso entre los mejores, uno de cada cinco se quedó sin equipo.

**Después, el draft.** ¿Los primeros en ser elegidos tienen mejores carreras? Cada punto es un jugador elegido: su número de selección y el valor de carrera que acumuló.

In [ ]:
#@title Gráfica 2
options(repr.plot.width = 12, repr.plot.height = 7)
if (hay_graficas) print(grafica_2) else sin_grafica()

En promedio, sí: la línea naranja cae con fuerza en las primeras selecciones. Pero mira la nube de puntos: en cada número de selección hay de todo. Peyton Manning fue la primera selección de 1998 y respondió; Tom Brady fue la 199 de 2000 y acumuló el mayor valor de carrera del archivo. **El número de selección ayuda a pronosticar la carrera, con enormes excepciones.**

**Por último, el pronóstico.** Juntamos lo que se sabe de cada jugador (posición, medidas y pruebas del Combine, estadísticas universitarias y, en el panel de la derecha, su número de selección) en un modelo estimado con los drafts de 2000 a 2010. Con él pronosticamos el valor de carrera de los elegidos de 2011 a 2014, que el modelo no vio. Si el pronóstico fuera perfecto, todos los puntos caerían sobre la línea discontinua.

In [ ]:
#@title Gráfica 3
options(repr.plot.width = 12, repr.plot.height = 8)
if (hay_graficas) print(grafica_3) else sin_grafica()

Con más información el pronóstico mejora: el error medio baja de 14.6 a 12.5 puntos cuando se conoce el número de selección. Pero los puntos siguen muy lejos de la línea. **El éxito se puede pronosticar solo en parte: siempre queda mucho ruido.** Entender por qué, y no prometer más de lo que los datos permiten, es buena parte del curso.

| | |
|--:|---|
| **14,163** | prospectos medidos en el Combine, de 1987 a 2024 |
| **59** | columnas: medidas físicas, pruebas, universidad, draft y carrera |
| **8,234** | fueron elegidos en el draft |
| **199** | el número de selección de Tom Brady, el jugador con más valor de carrera del archivo |

## Lo que harás en cada sesión

| Sesión | La pregunta | Lo que aprendes a hacer en R |
|--------|-------------|------------------------------|
| 1 | ¿Qué datos tenemos y cuáles sirven? | Cargar, mirar y limpiar una tabla |
| 2 | ¿Cómo se reparte el éxito y qué se relaciona con él? | Resumir, comparar grupos y calcular correlaciones |
| 3 | ¿Cómo se ve el problema? | Hacer gráficas que respondan preguntas |
| 4 | ¿Se puede pronosticar el éxito? | Estimar modelos y medir su error con jugadores que no vieron |
| 5 | ¿Qué tan bueno es el indicador y para qué sirve? | Reunir todo en un proyecto reproducible y comunicarlo con sus límites |

Lo harás programando en R, con ayuda de la IA si quieres, pero con una regla que no cambia en todo el curso: **la IA puede escribir código por ti; tú necesitas entender qué hace, comprobar si está bien y responsabilizarte de lo que presentas.**

Tres preguntas que vas a poder responder al final:

- ¿Por qué un buen Combine aumenta la probabilidad de ser elegido, pero no garantiza una buena carrera?
- ¿Cómo es posible que nadie eligiera a Tom Brady hasta la selección 199? ¿Lo habría visto venir algún modelo?
- ¿Sirve de algo un pronóstico que se equivoca, en promedio, por más de 12 puntos? Spoiler: depende de para qué lo uses.

*El código de las tres gráficas está en la celda «Preparar las gráficas de la portada», que aparece cerrada: haz doble clic sobre ella para verlo. Hoy no hace falta entenderlo.*

---

# Antes de empezar

Este cuaderno mezcla **celdas de texto**, como esta, y **celdas de código**, en R. Puedes leerlo como un documento y, al mismo tiempo, ejecutar cada celda para ver qué hace.

- Para ejecutar una celda: haz clic en ella y presiona `Shift + Enter` (o el botón ▶ a su izquierda). El resultado aparece debajo.
- Las celdas se ejecutan **en orden**: si una usa un objeto creado más arriba, ejecuta antes las anteriores (*Entorno de ejecución > Ejecutar anteriores*).
- Para ejecutar todo desde el principio: *Entorno de ejecución > Reiniciar y ejecutar todo*.

**Los paquetes.** Un *paquete* es un conjunto de funciones que alguien escribió y publicó para que no tengas que programarlas tú. En todo el curso usaremos solo tres: `dplyr` (manipular tablas), `ggplot2` (gráficas) y `glmnet` (un modelo de la sesión 4). La celda de preparación, al principio del cuaderno, instala los que falten. Lo que harás en cada cuaderno es *cargarlos* con `library()`.

**Tu archivo de trabajo.** Si todavía no lo hiciste, guarda una copia con *Archivo > Guardar una copia en Drive* y cámbiale el nombre a `sesion1_trabajo.ipynb` (clic en el nombre, arriba a la izquierda). Trabaja en esa copia: la subirás a GitHub en la tarea.

Los ejercicios de clase se corrigen en clase, en voz alta: no hay soluciones al final del documento. Inténtalos aunque te equivoques.

**Cómo se evalúa este curso.** En las tareas no basta con que el código se ejecute. Se evalúa que puedas explicar qué hace cada línea, que compruebes tus resultados con los datos y que justifiques tus decisiones, hayas escrito el código tú o te lo haya propuesto una IA. Varias tareas incluyen una decisión ya tomada (un filtro, un umbral, una variable, una forma de tratar los faltantes o de medir el error) para que la pongas a prueba: puede ser adecuada para la pregunta o no serlo. Lo que cuenta es que la compruebes con los datos, expliques sus consecuencias y, si hace falta, propongas una alternativa razonable.

## Qué vas a poder hacer al terminar

1. Cargar un archivo desde una URL indicando las opciones que su formato necesita, y explicar por qué.
2. Inspeccionar una tabla: dimensiones, nombres, tipos de variables y valores faltantes.
3. Seleccionar columnas, filtrar filas, convertir tipos y crear variables con `dplyr`, encadenando pasos con `|>`.
4. Distinguir entre éxito observado, variables predictoras y pronóstico.
5. Detectar errores en los datos con comprobaciones sencillas y documentar cada decisión de limpieza.
6. Explicar por qué dos códigos que funcionan, uno escrito por una persona y otro por una IA, pueden dar cifras distintas para la misma pregunta.

## Organización de la sesión (90 minutos)

| Minutos | Qué hacemos |
|---------|-------------|
| 0–15 | De qué trata el curso: el fútbol americano, el draft y la pregunta; éxito, predictores y pronóstico |
| 15–28 | Cargar los datos desde internet y mirarlos: objetos, funciones y tipos |
| 28–50 | De 14,163 filas a una base de análisis: convertir, filtrar, seleccionar y crear variables |
| 50–65 | Faltantes y comprobaciones: quién nunca jugó, nombres repetidos, valores raros |
| 65–80 | Actividad central: la versión normal y la versión IA |
| 80–90 | La receta completa; lo que sabemos y lo que no; GitHub |

Después de clase: **Tarea 1**, una decisión de limpieza que poner a prueba y la creación de tu repositorio de GitHub.

---

# 1. El encargo: éxito, predictores y pronóstico

Trabajas en el departamento de análisis de un equipo de la NFL. El director general te escribe:

> "Cada año elegimos a siete u ocho jugadores y la mitad no nos sirve. Quiero un indicador que, con lo que sabemos de un jugador el día del draft, nos diga qué carrera podemos esperar. Y quiero saber qué medidas valen la pena y cuáles son ruido."

Antes de tocar los datos, tres palabras que vamos a usar todo el curso, y que no son lo mismo:

| Concepto | Qué es | Ejemplo en este proyecto | ¿Cuándo se conoce? |
|----------|--------|--------------------------|--------------------|
| **Éxito observado** | Lo que efectivamente pasó en la carrera del jugador | El valor de carrera (`career_av`) o haber ido al Pro Bowl | Años después del draft |
| **Variables predictoras** | Lo que sabemos del jugador cuando hacemos el pronóstico | Estatura, peso, tiempo en 40 yardas, posición | En el Combine o el día del draft |
| **Pronóstico** | El valor o la probabilidad que estimamos a partir de los predictores | "Esperamos unos 25 puntos de valor de carrera" o "15% de probabilidad de ir al Pro Bowl" | Cuando tomamos la decisión |

La regla de oro: **un predictor solo vale si se conoce en el momento del pronóstico.** Los partidos que jugó un jugador en su carrera dicen mucho de su éxito, pero no sirven para pronosticarlo: el día del draft todavía no se han jugado.

Las dos medidas de éxito son las de la introducción: el valor de carrera (`career_av`) y haber ido al Pro Bowl. El valor de carrera es aproximado, como dice su nombre (*Approximate Value*), pero permite comparar jugadores de posiciones distintas en una sola escala. En la sesión 2 veremos sus ventajas y sus problemas.

## Ejercicio 1. Definir y anticipar (sin computadora)

a) Si fueras el director general, ¿cómo medirías que un jugador "tuvo éxito"? Propón dos medidas distintas y di qué desventaja tiene cada una.

b) Escribe tres datos de un jugador que **sí** conocerías el día del draft y tres que **no**.

c) Anticipa: ¿crees que el tiempo en 40 yardas ayudará a pronosticar el éxito? ¿Más o menos que la posición en la que fue elegido?

*Tus respuestas:*

a)

b)

c)

---

# 2. Cargar los datos desde internet

## Paquetes, objetos y funciones

Empezamos cargando `dplyr`, el paquete con el que vamos a manipular la tabla:

In [ ]:
library(dplyr)

Ahora guardamos la dirección del archivo en un **objeto**. El operador `<-` (se lee "asigna") guarda lo que está a la derecha con el nombre de la izquierda:

In [ ]:
url_datos <- paste0(
  "https://raw.githubusercontent.com/phcs971/nfl-draft-dataset/",
  "54730eaae524dbff0647e966cd12c739724f807f/nfl_data.csv"
)

`paste0()` es una **función**: recibe valores entre paréntesis (sus *argumentos*) y devuelve un resultado; esta en concreto pega textos. La usamos solo para que la dirección quepa en pantalla.

Dos detalles de esa dirección importan:

- Es el archivo `nfl_data.csv` del repositorio [phcs971/nfl-draft-dataset](https://github.com/phcs971/nfl-draft-dataset) (licencia MIT), que une resultados del Combine, datos del draft y estadísticas de carrera tomadas de Pro Football Reference y Sports Reference.
- Lleva un código largo (`54730ea...`): es el identificador de una **versión concreta** del archivo. Si el autor lo actualiza mañana, tu código seguirá leyendo exactamente los mismos datos. Eso es **reproducibilidad**: cualquiera que ejecute tu archivo obtiene tus mismas cifras.

## Leer el archivo: por qué hacen falta tres opciones

`read.csv()` lee archivos de texto con forma de tabla. Por defecto supone que las columnas están separadas por comas. Este archivo no:

In [ ]:
prueba <- try(read.csv(url_datos))

`try()` deja que el cuaderno siga aunque haya un error: así, si ejecutas todo de una vez, Colab muestra el error y continúa. El error (`duplicate 'row.names' are not allowed`) no dice "te equivocaste de separador", pero eso es lo que pasa. El archivo separa las columnas con **punto y coma** (lo indica su documentación), y algunas celdas contienen comas dentro del texto, como `"Tom Brady, Michigan"`. R cuenta mal las columnas y termina confundido. Lección: un error de R a veces describe el síntoma, no la causa. Cuando un archivo no se lee, lo primero es mirar su documentación o sus primeras líneas.

Leemos con tres opciones explícitas:

- `sep = ";"`: las columnas están separadas por punto y coma.
- `na.strings = ""`: una celda vacía significa "dato faltante" (`NA`). Sin esta opción, R deja las celdas de texto vacías como texto vacío (`""`), que parece un valor pero no lo es.

- `encoding = "UTF-8"`: le dice a R cómo están escritos los caracteres especiales. El archivo casi no los tiene, pero hay algunos: en la columna `college_id`, cinco filas llevan un guion largo, como `Wisconsin–Stout`. Sin esta opción, en algunas computadoras ese guion se vería como símbolos raros.

In [ ]:
ruta_respaldo <- "datos/nfl_data.csv"   # copia local, por si no hay internet

nfl <- tryCatch(
  read.csv(url_datos, sep = ";", na.strings = "", encoding = "UTF-8"),
  error = function(e) read.csv(ruta_respaldo, sep = ";", na.strings = "", encoding = "UTF-8")
)

`tryCatch()` se lee así: "intenta leer desde la URL; si hay un error (por ejemplo, no hay internet), lee la copia local". En la computadora, la copia está en la carpeta `datos/` del material del curso. En Colab casi nunca hace falta, porque siempre hay internet; si alguna vez fallara la URL, crea una carpeta `datos` en el panel de archivos (el ícono de la carpeta, a la izquierda) y sube ahí `nfl_data.csv`.

Primera comprobación, siempre: ¿llegó completo?

In [ ]:
dim(nfl)
if (nrow(nfl) != 14163 || ncol(nfl) != 59) {
  warning("El archivo no tiene 14,163 filas y 59 columnas: revisa la URL o la copia local.")
}

`dim()` devuelve filas y columnas. El `if` avisa si no coinciden con lo esperado; se lee: "si el número de filas es distinto (`!=`) de 14,163 o (`||`) el de columnas distinto de 59, avisa".

---

# 3. Mirar la tabla por dentro

## Nombres y tipos

In [ ]:
names(nfl)

Tres nombres llaman la atención: `X40_yard`, `X3_cone` y `X60yd_shuttle`. En el archivo original se llaman `40_yard`, `3_cone` y `60yd_shuttle`, pero en R un nombre no puede empezar con un número, así que `read.csv()` les antepone una `X`. No es un error: es R protegiéndose. Lo arreglaremos dándoles un nombre propio.

`glimpse()`, de `dplyr`, muestra cada columna con su tipo y sus primeros valores. Con 59 columnas es la forma más rápida de ver la tabla entera:

In [ ]:
glimpse(nfl)

Los tipos que aparecen:

| Tipo | Qué es | Ejemplo |
|------|--------|---------|
| `<int>` | Número entero | `year` |
| `<dbl>` | Número con decimales | `height`, `career_av` |
| `<chr>` | Texto | `name`, `pos`, y también `drafted` |
| `<lgl>` | Lógico: `TRUE` o `FALSE` | `X60yd_shuttle` |

Dos cosas raras, y las dos son información:

- `drafted` es **texto** (`"True"`/`"False"`), no un valor lógico. Así lo escribió quien armó el archivo. Lo convertiremos.
- `X60yd_shuttle` es **lógica** y todos sus valores son `NA`: la columna está vacía. Cuando R no encuentra ni un solo dato, no puede adivinar el tipo y elige el más simple.

## Lo que dice el archivo sobre sí mismo

In [ ]:
table(nfl$drafted)
summary(nfl$career_av)

El símbolo `$` saca una columna de la tabla. `table()` cuenta cuántas veces aparece cada valor; `summary()` resume una columna numérica. `career_av` tiene 6,866 `NA`: los 5,929 jugadores no elegidos no tienen carrera registrada en el archivo, y faltan otros 937. Averiguaremos quiénes son.

La documentación del archivo dice que las medidas están en **unidades estadounidenses**: estatura en pulgadas, peso en libras, saltos en pulgadas, tiempos en segundos. Compruébalo tú: ¿tiene sentido una estatura media de unas 74 pulgadas (1.88 m) y un peso de unas 240 libras (109 kg)?

In [ ]:
summary(nfl$height)
summary(nfl$weight)

# 4. De 14,163 filas a una base de análisis

Nuestra base de análisis tiene que contener a los jugadores para los que podemos medir el éxito y cuya carrera ya se puede evaluar. Vamos a construirla paso a paso, comprobando en cada paso cuántas filas quedan.

## Convertir un tipo y encadenar pasos

`mutate()` crea o modifica columnas. `as.logical()` convierte el texto `"True"`/`"False"` en los valores lógicos `TRUE`/`FALSE`:

In [ ]:
elegidos <- nfl |>
  mutate(drafted = as.logical(drafted)) |>
  filter(drafted)

nrow(elegidos)

El símbolo `|>` (*pipe*) se lee **"y luego"**: "toma `nfl`, y luego convierte `drafted`, y luego quédate con las filas en que `drafted` es verdadero". Cada función recibe la tabla que le pasa la anterior. `filter()` conserva las filas que cumplen una condición.

Fíjate en que guardamos el resultado en un objeto nuevo, `elegidos`, y no tocamos `nfl`: si nos equivocamos, el original sigue intacto.

## Filtrar: quiénes entran en el análisis

Solo los jugadores elegidos tienen datos de carrera en este archivo, así que el análisis se limita a ellos. Ahora, los años:

In [ ]:
elegidos <- elegidos |>
  filter(year >= 2000, year <= 2014)

nrow(elegidos)

Dentro de `filter()`, separar condiciones con comas significa "y": año mayor o igual que 2000 y menor o igual que 2014. ¿Por qué esos años?

- **Hasta 2014**: el archivo llega a la temporada 2024. Un jugador elegido en 2014 ha tenido al menos diez temporadas para acumular valor; uno elegido en 2022, solo tres. Si los mezcláramos, los jugadores recientes parecerían peores solo porque su carrera no ha terminado. En la sesión 2 lo comprobaremos con los datos.
- **Desde 2000**: es una NFL más parecida a la actual, con siete rondas de draft (hasta 1992 había doce) y con quince generaciones, suficientes para el proyecto.

Ahora las posiciones. `simple_pos` agrupa las posiciones detalladas:

In [ ]:
table(elegidos$simple_pos)

`ST` son los equipos especiales: pateadores (`K`) y despejadores (`P`). Casi no hacen las pruebas físicas y su valor de carrera se mide de otra manera, así que los dejamos fuera del análisis. El signo `!=` significa "distinto de":

In [ ]:
elegidos <- elegidos |>
  filter(simple_pos != "ST")

nrow(elegidos)

## Seleccionar y renombrar columnas

De 59 columnas necesitamos unas pocas. `select()` se queda con las que nombramos, en ese orden; `rename()` cambia un nombre (`nuevo = viejo`):

In [ ]:
mini <- elegidos |>
  select(name, year, simple_pos, draft_pick, weight, X40_yard, career_av, pro_bowls) |>
  rename(forty = X40_yard)

head(mini)

`head()` muestra las primeras seis filas. `forty` es más fácil de escribir que `X40_yard` y se entiende igual.

## Crear variables

Dos variables nuevas que usaremos en el proyecto:

- `pro_bowl`: `TRUE` si el jugador fue al menos una vez al Pro Bowl. Es una versión sí/no del éxito.
- `speed_score`: un indicador de velocidad que tiene en cuenta el peso, propuesto por el analista Bill Barnwell: `peso × 200 / tiempo⁴` (peso en libras, tiempo en segundos). Correr 4.5 segundos no tiene el mismo mérito pesando 180 libras que pesando 250. La fórmula está calibrada para que un corredor típico dé alrededor de 100.

In [ ]:
mini <- mini |>
  mutate(
    pro_bowl    = pro_bowls >= 1,
    speed_score = weight * 200 / forty^4
  )

head(mini)

`^` eleva a una potencia y `>=` significa "mayor o igual que": `pro_bowls >= 1` devuelve `TRUE` o `FALSE` para cada jugador.

**Comprueba que la variable mide lo que dice.** Si la fórmula está bien escrita, la mediana de los corredores (`RB`) debería rondar 100:

In [ ]:
mini |>
  filter(simple_pos == "RB") |>
  summarise(mediana_speed = median(speed_score, na.rm = TRUE))

`summarise()` resume una tabla en una fila. `na.rm = TRUE` le dice a `median()` que ignore los faltantes; sin él, un solo `NA` hace que el resultado sea `NA`. Sale cerca de 100: la fórmula está bien escrita. Si hubiéramos escrito `forty^2` en lugar de `forty^4`, el número saldría absurdo y lo habríamos notado aquí.

---

# 5. Faltantes y primeras comprobaciones

## ¿Qué falta?

`is.na()` pregunta "¿falta?" para cada celda y `colSums()` cuenta los `TRUE` de cada columna:

In [ ]:
elegidos |>
  select(height, weight, X40_yard, vert_leap, broad_jump, bench_press, shuttle,
         age, career_av, pro_bowls, g) |>
  is.na() |>
  colSums()

Tres lecturas:

- Las pruebas del Combine tienen bastantes faltantes: no todos los jugadores hacen todas las pruebas.
- `career_av` y `g` (partidos jugados) faltan para 309 jugadores **elegidos**. Eso no es lo mismo que los no elegidos.
- `age` falta para 180. Volveremos a esta columna en la sesión 2: sus faltantes esconden una sorpresa.

## ¿Quiénes son los elegidos sin carrera?

In [ ]:
elegidos |>
  filter(is.na(career_av), draft_round <= 3) |>
  select(year, name, pos, draft_round, draft_pick, pro_bowls, g)

Algunos nombres son conocidos entre los aficionados: Giovanni Carmazzi (2000) nunca jugó un partido oficial; Maurice Clarett (2005) tampoco; Chad Jones (2010) sufrió un accidente antes de debutar. El patrón es claro: **el archivo deja en blanco la carrera de los jugadores elegidos que nunca jugaron en la NFL**. Comprobémoslo con los 309:

In [ ]:
elegidos |>
  filter(is.na(career_av)) |>
  summarise(n = n(),
            con_pro_bowl = sum(pro_bowls >= 1),
            con_partidos = sum(!is.na(g)))

Ninguno fue al Pro Bowl ni tiene partidos registrados. **Decisión:** para estos jugadores, `career_av = 0`. Su carrera existió y valió cero. La alternativa, eliminarlos, borraría precisamente a los fracasos más caros y haría que el draft pareciera más acertado de lo que es. Dejamos además una columna `nunca_jugo` para no perder el rastro de la decisión.

## ¿El nombre identifica a un jugador?

La columna `tag` combina nombre y universidad. ¿Se repite alguna vez?

In [ ]:
sum(duplicated(nfl$tag))

tags_repetidos <- nfl$tag[duplicated(nfl$tag)]

nfl |>
  filter(tag %in% tags_repetidos) |>
  select(year, name, pos, drafted, team, draft_pick, year_max, career_av) |>
  arrange(name, year)

`duplicated()` marca cada valor que ya apareció antes; `%in%` pregunta "¿está en esta lista?"; `arrange()` ordena. Hay 8 parejas con el mismo nombre y la misma universidad en años distintos. Algunas son padre e hijo: Jon Runyan jugó en Michigan y fue elegido en 1996; su hijo, también Jon Runyan y también de Michigan, en 2020.

Ahora mira las columnas `year`, `pos`, `team` y `year_max` (último año en que jugó). Mario Edwards aparece como esquinero (CB) medido en el Combine de 2000, elegido por los Raiders en el puesto 35, y con una carrera que llega hasta 2024: **un esquinero con 25 temporadas**. No es verosímil. Lo que pasó es que el archivo unió los datos del draft y de la carrera usando nombre y universidad, y cuando había dos personas con el mismo `tag` pegó la información del hijo (elegido en 2015 como ala defensiva) en la fila del padre. Lo mismo ocurre con otras filas de la lista: la fila de Jon Runyan de 1996 lleva el equipo y el número de selección del hijo (Packers, 192) y una carrera que llega a 2024.

**Decisión:** excluimos todas las filas cuyo `tag` está repetido. En nuestro periodo son 3 jugadores elegidos (Mario Edwards 2000, Mike Davis 2009 y Jordan Jefferson 2012), cuya carrera registrada pertenece a otra persona. Para cada jugador elegido sí hay un identificador único, `player_id`, que comprobaremos al final.

## ¿Hay valores imposibles?

In [ ]:
summary(elegidos$career_av)

elegidos |>
  filter(career_av < 0) |>
  select(year, name, pos, draft_pick, career_av, g)

Un valor de carrera negativo parece un error, pero el AV puede ser negativo en temporadas muy malas, sobre todo de *quarterbacks*. Ryan Lindley (2012) jugó 10 partidos con muy malos números. Es un valor real: lo conservamos. **Comprobar no siempre significa corregir**; a veces significa entender.

---

# 6. La receta completa

Juntamos todos los pasos en un solo bloque. Esta **receta** es la que copiarás al principio de cada sesión, para que cada documento funcione solo, de principio a fin:

In [ ]:
tags_repetidos <- nfl$tag[duplicated(nfl$tag)]   # nombre + universidad repetidos

base <- nfl |>
  rename(forty = X40_yard) |>
  mutate(drafted = as.logical(drafted)) |>
  filter(drafted,                          # solo elegidos: los únicos con carrera registrada
         year >= 2000, year <= 2014,       # al menos diez temporadas para acumular valor
         simple_pos != "ST",               # sin pateadores ni despejadores
         !(tag %in% tags_repetidos)) |>    # sin filas con la carrera de otra persona
  mutate(nunca_jugo  = is.na(career_av),   # elegido, pero sin partidos en la NFL
         career_av   = ifelse(nunca_jugo, 0, career_av),
         g           = ifelse(nunca_jugo, 0, g),
         pro_bowl    = pro_bowls >= 1,
         speed_score = weight * 200 / forty^4) |>
  select(player_id, name, college, year, pos, simple_pos, draft_round, draft_pick, age,
         height, weight, hand_size, arm_length, forty, vert_leap, broad_jump,
         bench_press, shuttle, speed_score,
         career_av, pro_bowls, pro_bowl, g, years_as_primary_starter, nunca_jugo,
         college_games, college_pass_yds, college_pass_td, college_rush_yds,
         college_rec_yds, college_tackles, college_sacks, college_ints)

Las últimas columnas (`college_...`) son estadísticas de la etapa universitaria (NCAA): partidos jugados, yardas por pase, por tierra y por recepción, pases de anotación, tacleadas, capturas e intercepciones. Las usaremos desde la sesión 2.

Dos piezas nuevas: `!` niega una condición (`!(tag %in% tags_repetidos)` es "el tag **no** está en la lista") e `ifelse(condición, sí, no)` elige un valor u otro fila por fila.

Comprobaciones finales:

In [ ]:
nrow(base)                       # deberían ser 3,197
sum(duplicated(base$player_id))  # 0: cada fila es un jugador distinto
sum(is.na(base$career_av))       # 0: ya no faltan resultados
table(base$nunca_jugo)           # 309 elegidos que nunca jugaron

| Paso | Filas |
|------|-------|
| Archivo completo | 14,163 |
| Elegidos en el draft | 8,234 |
| Elegidos entre 2000 y 2014 | 3,232 |
| Sin pateadores ni despejadores | 3,200 |
| Sin `tag` repetido | 3,197 |

Esta tabla es la primera parte de la documentación de tu proyecto: cada fila que se fue tiene una razón escrita.

---

# 7. Actividad central: la versión normal y la versión IA

El director general hace una pregunta concreta:

> "¿Cuánto valor de carrera acumula, en promedio, un jugador que elegimos en el draft? Usa los drafts de 2000 a 2014."

## Paso 1. Anticipa

Con lo que viste en `summary(elegidos$career_av)`, ¿cuánto debería dar, aproximadamente? *Mi estimación:*

## Paso 2. La versión normal

> **Versión normal**
>

In [ ]:
mean(base$career_av)
median(base$career_av)

Usa la base que acabamos de construir, con sus decisiones documentadas, y da dos cifras: la media y la mediana.

## Paso 3. La versión IA

La respuesta del asistente fue **redactada para este curso** imitando la forma en que suelen contestar estas herramientas; no es la transcripción de una herramienta concreta. El código sí se ejecuta con tus datos.

> **Tú**
>
> Tengo en R un data frame `nfl` con datos del NFL Combine y del draft (columnas `drafted`, `year`, `career_av`, `X40_yard`, `weight`...). ¿Cuánto valor de carrera (`career_av`) acumula en promedio un jugador elegido en el draft entre 2000 y 2014?

> **Asistente de IA**
>
> ¡Excelente pregunta! 🏈 Aquí tienes un análisis **robusto y reproducible** con `dplyr`, siguiendo buenas prácticas de limpieza de datos:
>

In [ ]:
library(dplyr)

# ==========================================================
# Análisis del valor de carrera (AV) de jugadores drafteados
# ==========================================================

# --- Paso 1: Filtrar jugadores drafteados (2000-2014) -----
df_draft <- nfl %>%
  filter(drafted == "True", year >= 2000 & year <= 2014)

# --- Paso 2: Limpieza de datos ----------------------------
# Eliminamos registros con valores faltantes en las variables
# clave para evitar sesgos y garantizar resultados confiables
df_clean <- df_draft %>%
  filter(!is.na(career_av), !is.na(X40_yard), !is.na(weight))

# --- Paso 3: Estadísticas descriptivas --------------------
resumen_av <- df_clean %>%
  summarise(
    n_jugadores = n(),
    av_promedio = mean(career_av),
    av_mediana  = median(career_av),
    av_desv_est = sd(career_av),
    av_maximo   = max(career_av)
  )
print(resumen_av)

# --- Paso 4: Resultado ------------------------------------
cat(sprintf("✅ Un jugador drafteado acumula en promedio %.1f puntos de AV.\n",
            resumen_av$av_promedio))

>
> **Explicación del código:**
>
> 1. **Filtrado:** selecciona a los jugadores elegidos en el draft entre 2000 y 2014.
> 2. **Limpieza:** elimina los registros incompletos, lo que evita sesgos y garantiza que el promedio sea representativo.
> 3. **Estadísticas:** calcula la media, la mediana, la desviación estándar y el máximo para una visión completa.
>
> 💡 **Interpretación:** un jugador drafteado típico acumula alrededor de 21 puntos de AV a lo largo de su carrera, lo que refleja el valor que los equipos obtienen en promedio de cada selección.
>
> ¿Quieres que también lo desglose por ronda o por posición? 😊

Funciona, limpia los datos, comenta cada paso y termina con un mensaje seguro de sí mismo. Pero no da lo mismo que la versión normal.

## Paso 4. ¿En qué se nota que no lo escribió una persona?

En parejas, encuentren al menos **cuatro rasgos** de la versión IA que una persona rara vez escribiría para responder una pregunta de una línea (fíjense en los comentarios, en lo que afirma, en cómo presenta el resultado). Para cada uno, decidan si **ayuda**, **estorba** o **da igual**.

| Rasgo de la versión IA | ¿Ayuda, estorba o da igual? ¿Por qué? |
|------------------------|----------------------------------------|
| | |
| | |
| | |
| | |

Que un código parezca escrito por una IA no lo hace incorrecto, y que esté bien presentado no lo hace correcto. Lo que decide es qué hace con los datos.

## Paso 5. Lee antes de juzgar

Siguen en parejas, unos 10 minutos.

a) Para cada línea de código de la versión IA, escribe qué hace. `%>%` es otra forma de escribir `|>`; hacen lo mismo.

b) Completa:

| | Versión normal | Versión IA |
|---|---|---|
| ¿Con cuántos jugadores calcula? | | |
| Media | | |
| Mediana | | |

c) ¿De dónde sale la diferencia? Estas instrucciones te ayudan:

In [ ]:
sum(is.na(df_draft$career_av))
sum(is.na(df_draft$X40_yard))

base |> filter(is.na(forty))  |> summarise(n = n(), av_medio = mean(career_av))
base |> filter(!is.na(forty)) |> summarise(n = n(), av_medio = mean(career_av))

table(df_clean$simple_pos)

d) La explicación dice que eliminar los registros incompletos "evita sesgos". Con lo que encontraste en c), ¿los evita o los crea? ¿Hacia dónde empuja el promedio?

e) La interpretación habla de un jugador "típico" con 21 puntos. Con la mediana a la vista, ¿la media describe a un jugador típico?

f) ¿Qué cifra le presentarías al director general y cómo la explicarías en dos frases?

> **Si tienes acceso a una IA.** Pídele lo mismo, pero añade lo que sabes: *"Los jugadores elegidos que nunca jugaron tienen `career_av` vacío; cuentan como 0. No elimines filas sin decirme cuántas y por qué."* Compara su nueva respuesta con la versión normal. Después pregúntale: *"¿Tu código elimina alguna fila? ¿Cuántas?"*, y compruébalo tú con `nrow()`.
>
> **Sin IA**, el ejercicio es el mismo: la conversación incluida en este documento cumple ese papel.

# 8. Ejercicio individual

## Ejercicio 3. Modificar la versión normal

a) Elige una posición (`simple_pos`: `QB`, `RB`, `WR`, `TE`, `OL`, `DL`, `LB` o `DB`). Calcula la media y la mediana de `career_av` solo para esa posición. Pista: `filter()` y después `summarise()`.

b) ¿Qué proporción de jugadores de la base fue al menos una vez al Pro Bowl? Pista: `mean(base$pro_bowl)`. La media de una columna de `TRUE`/`FALSE` es la proporción de `TRUE`, porque R cuenta `TRUE` como 1 y `FALSE` como 0.

c) Debajo de tu código, escribe un comentario de una línea con lo que cambiaste respecto a la versión normal y qué efecto tuvo.

In [ ]:
# a)

# b)

## Ejercicio 4. La ficha de la base

a) ¿Cuántos jugadores hay de cada año de draft? ¿Algún año se ve distinto?

b) De las pruebas del Combine (`forty`, `vert_leap`, `broad_jump`, `bench_press`, `shuttle`), ¿cuál tiene más faltantes en `base`?

c) Busca a Tom Brady y a Russell Wilson con `filter(name %in% c("Tom Brady", "Russell Wilson"))` y muestra su año, posición, número de selección, tiempo en 40 yardas y `career_av`. ¿Qué te dicen estos dos casos sobre la dificultad de pronosticar?

d) Busca a J.J. Watt, uno de los mejores defensivos de su generación. Si `filter(name == "J.J. Watt")` no devuelve nada, prueba `filter(grepl("Watt", name))`: `grepl()` busca un texto **dentro** de otro. ¿Qué aprendes sobre buscar por nombre?

In [ ]:
# Tu código

# 9. Lo que sabemos y lo que todavía no sabemos

Vuelve a tu Ejercicio 1. ¿Esperabas que una de cada diez selecciones no jugara nunca?

| Lo que sabemos de la base | Lo que todavía no sabemos |
|---------------------------|---------------------------|
| | |
| | |
| | |

---

# 10. Para cerrar: dónde va a vivir tu proyecto

Dos preguntas para discutir en grupo:

1. Si tuvieras que decir en una reunión "un jugador elegido acumula 18 puntos de valor de carrera en promedio", ¿qué tendrías que aclarar sobre quiénes están en ese promedio?

2. La versión IA tomó una decisión de limpieza en una línea y la explicó con una frase convincente. ¿Cómo te habrías dado cuenta sin la versión normal al lado?

Hoy tomaste cinco decisiones que cambian quién está en el análisis. Alguien tiene que poder ver **qué código se usó, cuándo se cambió y por qué**. Para eso usaremos **GitHub**: un sitio web donde guardarás tu proyecto en un *repositorio* que conserva cada versión que subes.

Tu repositorio será **público**. Así cualquiera puede revisar tu trabajo con solo abrir el enlace, y al terminar el curso tendrás un portafolio con un proyecto de pronóstico real, su código y la historia de cómo lo hiciste.

| Acción | Dónde queda | Quién lo ve | ¿Puedes recuperar la versión anterior? |
|--------|-------------|-------------|----------------------------------------|
| **Guardar** el cuaderno en Colab (`Ctrl + S`) | En tu Google Drive | Solo tú | Solo de forma limitada: no es un historial que otros puedan revisar |
| **Subir** el archivo a GitHub | En tu repositorio | Cualquiera que lo abra | Sí, porque cada subida queda registrada |
| **Registrar una versión** en el historial | En el historial del repositorio | Cualquiera que lo abra | Es justamente para eso: cada versión con su fecha y un mensaje |

Cuando subes archivos desde la página web de GitHub, las dos últimas acciones ocurren a la vez. Guardar en tu Drive no cambia nada en GitHub.

---

# Tarea 1 (después de clase): una decisión de limpieza y tu repositorio

> **Tiempo estimado:** 15–20 minutos para la parte A (en Colab) y 30–45 minutos para la parte B (en GitHub).
>
> **Qué entregas:** el enlace a tu repositorio público `curso-programacion-r`, con un `README.md` que documente la base de datos, una carpeta por sesión y tu cuaderno `sesion1_trabajo.ipynb` (con la parte A al final) dentro de `sesion-1/`, acompañado de una nota breve.
>
> **Cómo lo entregas:** pegas el enlace en la actividad «Tarea 1» de la plataforma del curso (paso 10).
>
> **Qué se valora:** que compruebes con los datos lo que calculas, que expliques tus decisiones y que tu repositorio permita revisar tu trabajo. Que el código se ejecute es necesario, pero no suficiente.

## Parte A. "Sin faltantes en el modelo"

Hazla en tu cuaderno `sesion1_trabajo.ipynb`, en una sección nueva al final: una celda de texto con el título `# Tarea 1: sin faltantes`. Añade celdas de código con **+ Código** y celdas de texto con **+ Texto** (arriba a la izquierda), y escribe tus respuestas en celdas de texto debajo de cada celda de código.

El analista jefe te escribe:

> "Para el modelo quiero una base limpia, sin un solo faltante. Quédate solo con los jugadores que tienen las cinco pruebas del Combine completas: 40 yardas, salto vertical, salto largo, press de banca y *shuttle*. Así nadie podrá decir que inventamos datos."

**1. Anticipa.** Antes de ejecutar nada: ¿cuántos jugadores de los 3,197 crees que quedarán? ¿Cambiará el tipo de jugador que queda?

**2. Implementa la propuesta tal como está:**

```r
base_completa <- base |>
  filter(!is.na(forty), !is.na(vert_leap), !is.na(broad_jump),
         !is.na(bench_press), !is.na(shuttle))
nrow(base_completa)
```

**3. Comprueba.** Compara `base` y `base_completa`: número de jugadores por posición (`table(...$simple_pos)`), media y mediana de `career_av`, proporción de Pro Bowl y número de jugadores de primera ronda (`sum(...$draft_round == 1)`). Presenta lo que encuentres en una tabla breve.

**4. Evalúa la decisión.** En tres o cuatro frases: ¿es adecuada esta base para pronosticar el éxito de **cualquier** jugador elegido? ¿Qué consecuencias tendría usarla? ¿Qué alternativa propones? Implementa tu alternativa y di cuántos jugadores conserva.

**5. Seguimiento.**

a) Una posición casi desaparece con la propuesta. ¿Cuál, y qué prueba lo provoca? Busca una explicación razonable de por qué esos jugadores no la hacen.

b) Si quitas la condición de `bench_press`, ¿cuántos jugadores recuperas? ¿Cambia tu conclusión?

## Parte B. Tu repositorio en GitHub

> **Antes de empezar: tu repositorio será público.** Cualquiera en internet podrá leerlo, y lo que subes queda en el historial aunque después lo borres. Nunca deben aparecer en él contraseñas, tokens o claves de acceso, códigos de verificación ni datos personales innecesarios (teléfono, domicilio, matrícula, fecha de nacimiento). Tu nombre basta. Si quieres que tu correo no aparezca en los cambios, activa en GitHub *Settings > Emails > Keep my email addresses private*.

## Paso 1. Crea tu cuenta de GitHub (si no tienes una)

1. Entra a <https://github.com> y pulsa **Sign up**.
2. Escribe tu correo, una contraseña y un **nombre de usuario**. El nombre de usuario será público y formará parte de la dirección de tu repositorio: elige uno sobrio (por ejemplo, tu nombre y apellido sin espacios). Piensa que puede acabar en tu currículum.
3. Completa la verificación que te pida GitHub (un rompecabezas y un código que llega a tu correo).
4. Si GitHub te pide activar la **verificación en dos pasos**, sigue sus instrucciones y guarda los códigos de recuperación en un lugar seguro, fuera del repositorio.

**Cómo saber que funcionó:** al entrar a <https://github.com> ves tu imagen de perfil (o un ícono) en la esquina superior derecha.

## Paso 2. Crea el repositorio

1. Pulsa el botón **+** de la esquina superior derecha y elige **New repository** (o entra a <https://github.com/new>).
2. En **Owner** debe aparecer tu nombre de usuario.
3. En **Repository name** escribe exactamente `curso-programacion-r`: minúsculas, guiones, sin acentos ni espacios. Si escribes espacios o acentos, GitHub los cambiará por guiones y el nombre ya no coincidirá con el que usan todas las instrucciones del curso.

**Cómo saber que funcionó:** GitHub indica junto al campo si el nombre está disponible.

## Paso 3. Añade una descripción

En **Description** escribe una frase como: *Proyecto de R: ¿qué características de un jugador permiten pronosticar su éxito en la NFL?*

## Paso 4. Déjalo público

En la opción de visibilidad elige **Public**. Cualquiera podrá verlo, pero solo tú podrás modificarlo.

## Paso 5. Crea el repositorio con un `README.md` que documente los datos

1. Activa la opción para añadir un README (*Add a README file* o *Add README*). Deja lo demás como está.
2. Pulsa **Create repository**.
3. Abre `README.md`, pulsa el lápiz (*Edit this file*) y reemplaza el contenido por esta plantilla, **completada con tus palabras**:

```
# Pronosticar el éxito en la NFL

Proyecto del curso "Programar con criterio en la era de la IA".
Pregunta: ¿qué características de un jugador, conocidas en el draft,
permiten pronosticar su éxito profesional?

Autor/a: [tu nombre y apellido]

## Datos
- Archivo: nfl_data.csv del repositorio phcs971/nfl-draft-dataset
  (licencia MIT), leído desde una URL fijada a la versión 54730ea.
- Contenido: [qué contiene, cuántas filas y columnas]
- Base de análisis: [quiénes entran y cuántos son]

## Variables principales
- Éxito observado: [variable y qué mide]
- Predictores: [variables y por qué se conocen en el draft]

## Decisiones de limpieza
1. [decisión y razón]
2. ...

## Carpetas
- sesion-1/ a sesion-5/: trabajos de cada sesión
```

4. Pulsa **Commit changes...**, escribe en **Commit message** algo descriptivo, como *Documentar datos y decisiones de limpieza*, y confirma en la rama `main`.

**Cómo saber que funcionó:** la portada del repositorio muestra tu README debajo de la lista de archivos.

## Paso 6. Cuatro palabras que vas a usar todo el curso

- **Repositorio**: la carpeta en línea, con dirección propia: `https://github.com/tu-usuario/curso-programacion-r`.
- **Archivo**: cada documento dentro del repositorio, organizado en carpetas.
- **Cambio** (*commit*): cada vez que creas, subes, editas o borras archivos y confirmas, GitHub registra un cambio con fecha, autor y mensaje.
- **Historial**: la lista de todos los cambios. Encima de la lista de archivos, el enlace con un reloj y la palabra *Commits*. Para un solo archivo: ábrelo y pulsa **History**.

**Comprueba:** abre el historial. Haz clic en la edición del README: GitHub muestra en verde lo que añadiste y en rojo lo que quitaste.

## Paso 7. Crea la estructura de carpetas

```
curso-programacion-r/
├── README.md
├── sesion-1/
├── sesion-2/
├── sesion-3/
├── sesion-4/
└── sesion-5/
```

No hace falta subir los datos: tu código los lee de la URL fijada. GitHub no guarda carpetas vacías, así que cada carpeta se crea con un archivo de notas:

1. En la portada, **Add file > Create new file**.
2. En el nombre escribe `sesion-1/notas.md`. Al escribir la diagonal `/`, GitHub convierte `sesion-1` en una carpeta.
3. En el contenido escribe `# Sesión 1`.
4. Pulsa **Commit changes...**, escribe un mensaje como *Crear carpeta de la sesión 1* y confirma.
5. Repite para `sesion-2` a `sesion-5`.

**Cómo saber que funcionó:** en la portada ves `README.md` y las cinco carpetas.

## Paso 8. Sube tu primer archivo de trabajo

**8.1 Comprueba que funciona de principio a fin.** En Colab, *Entorno de ejecución > Reiniciar y ejecutar todo*. Revisa que ninguna celda termine con un error que no sea a propósito y que la parte A esté completa, con sus resultados.

**8.2 Descárgalo.** *Archivo > Descargar > Descargar .ipynb*. Si el archivo no se llama `sesion1_trabajo.ipynb`, cámbiale el nombre.

**8.3 Súbelo.** En GitHub, entra a `sesion-1`, **Add file > Upload files** y arrastra `sesion1_trabajo.ipynb`. Mensaje: *Agregar trabajo de la sesión 1*. **Commit changes**.

**8.4 Escribe tu nota de entrega.** Edita `sesion-1/notas.md` con esto, completado con tus palabras:

```
# Sesión 1

1. Lo que quería conseguir:
2. Archivos y código: sesion1_trabajo.ipynb (carga nfl_data.csv, construye
   la base de análisis y, al final, la parte A)
3. Comprobaciones que hice: (filas en cada paso, ¿se ejecutó todo?, ¿con qué
   comprobaste la propuesta del analista jefe?)
4. Cambios que conservé o descarté: (qué hiciste con la propuesta y por qué)
5. Lo que puedo afirmar: dos o tres frases sobre la base de análisis
```

Confirma con un mensaje como *Nota de entrega de la sesión 1*.

## Paso 9. Comprueba que todo está bien

1. Abre `sesion-1/sesion1_trabajo.ipynb` en GitHub. GitHub muestra el cuaderno con el texto, el código y los resultados, tal como lo guardaste. Busca una de tus respuestas para confirmar que subiste **tu** versión.
2. Abre el historial (*Commits*): un cambio por paso, cada uno con su mensaje.
3. Abre tu repositorio en una ventana privada del navegador (sin sesión iniciada): eso es lo que ve cualquier persona. Si aparece un error 404, quedó privado. Corrígelo en **Settings**, zona roja *Danger Zone*, **Change visibility**, **Public**.

## Paso 10. Entrega el enlace

1. Copia la dirección de la portada de tu repositorio: `https://github.com/tu-usuario/curso-programacion-r`.
2. Pégala en la actividad «Tarea 1» de la plataforma del curso. Antes, pruébala en la ventana privada.

## Errores habituales y cómo resolverlos

| Lo que pasa | Por qué suele pasar | Qué hacer |
|-------------|---------------------|-----------|
| Todas las celdas de código dan `SyntaxError` | El cuaderno se abrió con Python | *Entorno de ejecución > Cambiar tipo de entorno de ejecución > R* |
| `there is no package called 'dplyr'` | No se ejecutó la celda de preparación | Ejecuta la primera celda de código del cuaderno |
| Error al leer la URL | Un fallo momentáneo de la conexión | Vuelve a ejecutar la celda; si sigue fallando, sube `nfl_data.csv` a una carpeta `datos` en el panel de archivos |
| `object 'base' not found` al ejecutar una celda suelta | No ejecutaste antes las celdas anteriores, o Colab se reinició por inactividad | *Entorno de ejecución > Ejecutar anteriores* |
| El repositorio se llama distinto | Se escribió con acentos o mayúsculas | **Settings > General > Repository name**, corrige y pulsa *Rename* |
| Subí el archivo a la carpeta equivocada | Se pulsó *Upload files* desde la portada | Abre el archivo, lápiz, y en el nombre escribe la carpeta delante (`sesion-1/...`) |
| En la ventana privada el enlace da *404* | El repositorio quedó privado o el enlace tiene una errata | Paso 9.3 |
| GitHub rechaza la subida porque contiene un secreto | Hay una clave o token en el archivo | Elimínalo antes de volver a subirlo |

## Lista de comprobación final

- [ ] El repositorio se llama `curso-programacion-r`, es público y tiene descripción.
- [ ] El `README.md` documenta el archivo de datos, la base de análisis, las variables principales y las decisiones de limpieza.
- [ ] Existen las carpetas `sesion-1` a `sesion-5`.
- [ ] `sesion-1/sesion1_trabajo.ipynb` es mi versión, con mis respuestas, y se ejecuta de principio a fin.
- [ ] La parte A está al final: la propuesta, mi comprobación, mi evaluación, mi alternativa y el seguimiento.
- [ ] `sesion-1/notas.md` tiene las cinco partes.
- [ ] Los mensajes de los cambios describen lo que hice.
- [ ] No hay contraseñas, tokens ni datos personales innecesarios.
- [ ] Pegué el enlace en la actividad «Tarea 1» de la plataforma del curso.

## Cómo se evalúa

| Criterio | Qué se revisa |
|----------|---------------|
| Organización | Nombre, descripción, README que documenta datos y decisiones, estructura de carpetas |
| Archivo de trabajo | Está en `sesion-1/`, compila y contiene tus respuestas de clase |
| Parte A | Pones a prueba la propuesta con los datos, explicas sus consecuencias, propones y calculas una alternativa justificada y respondes el seguimiento razonando |
| Trazabilidad | El historial muestra los pasos con mensajes que se entienden |
| Nota de entrega | Las cinco partes están completas |
| Cuidado | No hay información sensible y el repositorio se abre sin iniciar sesión |

No se evalúa la rapidez ni el dominio de Git: se evalúa que el repositorio se pueda revisar y que puedas explicar y defender lo que hay en tu archivo.